In [1]:
#loading the data using pandas

import pandas as pd

In [2]:
import os

os.getcwd()

'C:\\Users\\Shaik\\data_preparation'

In [3]:
df = pd.read_csv("data_set/loan_data.csv")
df.head(4)
                

,applicant_id,name,age,income,credit_score,loan_amount,employment_status,married,gender,approved
0,APP1170,Raj Smith,NaN,"$66,400",NaN,89100.0,Unemployed,No,Female,Rejected
1,APP1177,Omar Silva,25.0,"$75,700",645.0,228600.0,Employed,NaN,Male,Approved
2,APP1132,Mei Khan,24.0,"$26,400",580.0,5000.0,employed,Yes,Female,Approved
3,APP1193,Liam Ali,28.0,"$78,500",NaN,272600.0,Employed,No,Female,Approved


In [4]:
df.shape

(212, 10)

#### quick structural health check of your DataFrame

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 212 entries, 0 to 211
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   applicant_id       212 non-null    object 
 1   name               212 non-null    object 
 2   age                194 non-null    float64
 3   income             202 non-null    object 
 4   credit_score       195 non-null    float64
 5   loan_amount        212 non-null    float64
 6   employment_status  201 non-null    object 
 7   married            200 non-null    object 
 8   gender             202 non-null    object 
 9   approved           212 non-null    object 
dtypes: float64(3), object(7)
memory usage: 16.7+ KB


#### for more detailed info we use describe

In [6]:
#gives us statistical information
df.describe()

,age,credit_score,loan_amount
count,194.000000,195.000000,212.000000
mean,43.448454,648.574359,152261.792453
std,20.945523,132.753971,85665.271670
min,-5.000000,0.000000,5000.000000
25%,31.000000,582.500000,88950.000000
50%,41.000000,652.000000,155800.000000
75%,54.000000,737.500000,207575.000000
max,200.000000,850.000000,448800.000000


#### to look for null values

In [7]:
df.isna().sum()

applicant_id          0
name                  0
age                  18
income               10
credit_score         17
loan_amount           0
employment_status    11
married              12
gender               10
approved              0
dtype: int64

In [8]:
df.columns

Index(['applicant_id', 'name', 'age', 'income', 'credit_score', 'loan_amount',
       'employment_status', 'married', 'gender', 'approved'],
      dtype='object')

In [9]:
df.head(10)

,applicant_id,name,age,income,credit_score,loan_amount,employment_status,married,gender,approved
0,APP1170,Raj Smith,NaN,"$66,400",NaN,89100.0,Unemployed,No,Female,Rejected
1,APP1177,Omar Silva,25.0,"$75,700",645.0,228600.0,Employed,NaN,Male,Approved
2,APP1132,Mei Khan,24.0,"$26,400",580.0,5000.0,employed,Yes,Female,Approved
3,APP1193,Liam Ali,28.0,"$78,500",NaN,272600.0,Employed,No,Female,Approved
4,APP1102,Raj Kumar,35.0,NaN,573.0,159000.0,employed,N,Female,Rejected
5,APP1182,Raj Silva,26.0,"$12,000",574.0,78000.0,Unemployed,Y,Male,Rejected
6,APP1189,Liam Smith,37.0,"$63,200",692.0,5000.0,employed,Yes,male,Approved
7,APP1073,Sara Brown,NaN,"$104,000",784.0,170200.0,employed,no,Male,Approved
8,APP1051,John Smith,56.0,"$60,500",780.0,182400.0,employed,Y,M,Approved
9,APP1185,Nina Silva,23.0,"$76,700",NaN,13700.0,Employed,Yes,F,Approved


#### We are traing a model to predict loan approval do we don't need applicant_id, name (there is no use with this 2, we don't give loans based on names)

In [10]:
df= df.drop(["applicant_id", "name"], axis=1)
df.head(5)

# df.drop(columns= ["applicant_id", "name"]) --->by using columns= 
# df.drop([0, 1]) ----> by index of column

,age,income,credit_score,loan_amount,employment_status,married,gender,approved
0,NaN,"$66,400",NaN,89100.0,Unemployed,No,Female,Rejected
1,25.0,"$75,700",645.0,228600.0,Employed,NaN,Male,Approved
2,24.0,"$26,400",580.0,5000.0,employed,Yes,Female,Approved
3,28.0,"$78,500",NaN,272600.0,Employed,No,Female,Approved
4,35.0,NaN,573.0,159000.0,employed,N,Female,Rejected


In [11]:
#finding out duplicated rows

duplicate_rows= df[df.duplicated(keep=False)]
print(duplicate_rows)

      age   income  credit_score  loan_amount employment_status married  \
3    28.0  $78,500           NaN     272600.0          Employed      No   
5    26.0  $12,000         574.0      78000.0        Unemployed       Y   
13   47.0  $56,600         627.0     249000.0          employed      no   
16   60.0  $57,700         806.0     117600.0     self employed     NaN   
30   55.0  $47,600         632.0     113300.0          employed     Yes   
38    NaN  $61,300         745.0      70700.0     Self-Employed     yes   
60   36.0    34100         647.0     277800.0          employed      No   
64    NaN  $61,300         745.0      70700.0     Self-Employed     yes   
69   50.0  $26,300         624.0     129000.0     Self Employed     Yes   
73   28.0  $78,500           NaN     272600.0          Employed      No   
94    NaN  $12,000         786.0     126300.0          employed      no   
106  36.0    34100         647.0     277800.0          employed      No   
118  39.0  $62,500       

In [12]:
#drop duplicates

before= df.shape[0]
df= df.drop_duplicates()
after= df.shape[0]
print(f"Before: {before}, After: {after}")

Before: 212, After: 200


In [13]:
#converting income column into numerical column there are many ways to do that we are going to look some of it

df['income']= (
    df["income"].astype(str).str.replace(r"[$,]", "", regex= True).str.strip()
)
df.head(3)

,age,income,credit_score,loan_amount,employment_status,married,gender,approved
0,NaN,66400,NaN,89100.0,Unemployed,No,Female,Rejected
1,25.0,75700,645.0,228600.0,Employed,NaN,Male,Approved
2,24.0,26400,580.0,5000.0,employed,Yes,Female,Approved


In [14]:
#converting to numeric
df['income']= pd.to_numeric(df['income'], errors="coerce")
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 200 entries, 0 to 211
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   age                184 non-null    float64
 1   income             188 non-null    float64
 2   credit_score       184 non-null    float64
 3   loan_amount        200 non-null    float64
 4   employment_status  190 non-null    object 
 5   married            190 non-null    object 
 6   gender             190 non-null    object 
 7   approved           200 non-null    object 
dtypes: float64(4), object(4)
memory usage: 14.1+ KB


#### Now look at gender column it as different representation

In [15]:
df["gender"].unique()  #we want to a standard form of representation

array(['Female', 'Male', 'male', 'M', 'F', nan, 'female'], dtype=object)

In [16]:
df.gender.value_counts()

gender
Male      37
M         34
female    34
male      31
F         28
Female    26
Name: count, dtype: int64

In [17]:
#using maping fuction
df['gender']= df['gender'].map({
    'M':'Male', 'male': 'Male', 'F': 'Female', 'female': 'Female'
})

In [18]:
df.gender.unique()

array([nan, 'Male', 'Female'], dtype=object)

In [19]:
df['married'].unique()

array(['No', nan, 'Yes', 'N', 'Y', 'no', 'yes'], dtype=object)

In [20]:
#same with married column

df['married']= df['married'].map({
    'Y': 'Yes', 'yes': 'Yes',
    'no': 'No', 'N': 'No'
})

In [21]:
df['married'].unique()

array([nan, 'No', 'Yes'], dtype=object)

In [22]:
df.head(10)

,age,income,credit_score,loan_amount,employment_status,married,gender,approved
0,NaN,66400.0,NaN,89100.0,Unemployed,NaN,NaN,Rejected
1,25.0,75700.0,645.0,228600.0,Employed,NaN,NaN,Approved
2,24.0,26400.0,580.0,5000.0,employed,NaN,NaN,Approved
3,28.0,78500.0,NaN,272600.0,Employed,NaN,NaN,Approved
4,35.0,NaN,573.0,159000.0,employed,No,NaN,Rejected
5,26.0,12000.0,574.0,78000.0,Unemployed,Yes,NaN,Rejected
6,37.0,63200.0,692.0,5000.0,employed,NaN,Male,Approved
7,NaN,104000.0,784.0,170200.0,employed,No,NaN,Approved
8,56.0,60500.0,780.0,182400.0,employed,Yes,Male,Approved
9,23.0,76700.0,NaN,13700.0,Employed,NaN,Female,Approved


In [23]:
#same with employment_status

df.employment_status.value_counts()

employment_status
employed         61
Employed         49
Self-Employed    20
Unemployed       17
unemployed       15
self employed    14
Self Employed    14
Name: count, dtype: int64

In [24]:
df['employment_status']= df['employment_status'].map({
    'employed': 'Employed', 'Self-Employed': 'Self_Employed', 'self employed': 'Self_Employed', 'Self employed': 'Self_Employed', 'unemployed': 'Unemployed'
})

In [25]:
df.employment_status.value_counts()

employment_status
Employed         61
Self_Employed    34
Unemployed       15
Name: count, dtype: int64

#### Dealing with outliers

In [26]:
df.describe()

,age,income,credit_score,loan_amount
count,184.000000,188.000000,184.000000,200.000000
mean,43.527174,52789.361702,645.951087,151629.500000
std,21.366674,21143.955119,134.420646,86473.742375
min,-5.000000,12000.000000,0.000000,5000.000000
25%,31.000000,37925.000000,581.500000,88250.000000
50%,41.000000,51150.000000,656.000000,157450.000000
75%,54.000000,68325.000000,729.250000,205700.000000
max,200.000000,104000.000000,850.000000,448800.000000


In [27]:
#look there are outliers in age min= -5.000000 and max= 200.000000 and loan age is 18 so setting condition for that

import numpy as np
import pandas as pd
df.loc[(df['age'] < 18) | (df['age'] > 100), 'age']= np.nan


In [28]:
df.head()

,age,income,credit_score,loan_amount,employment_status,married,gender,approved
0,NaN,66400.0,NaN,89100.0,NaN,NaN,NaN,Rejected
1,25.0,75700.0,645.0,228600.0,NaN,NaN,NaN,Approved
2,24.0,26400.0,580.0,5000.0,Employed,NaN,NaN,Approved
3,28.0,78500.0,NaN,272600.0,NaN,NaN,NaN,Approved
4,35.0,NaN,573.0,159000.0,Employed,No,NaN,Rejected


In [29]:
df.describe()

,age,income,credit_score,loan_amount
count,180.000000,188.000000,184.000000,200.000000
mean,42.327778,52789.361702,645.951087,151629.500000
std,12.861458,21143.955119,134.420646,86473.742375
min,21.000000,12000.000000,0.000000,5000.000000
25%,31.000000,37925.000000,581.500000,88250.000000
50%,41.000000,51150.000000,656.000000,157450.000000
75%,54.000000,68325.000000,729.250000,205700.000000
max,64.000000,104000.000000,850.000000,448800.000000


In [30]:
#by same logic for credit score
df.loc[(df['credit_score'] < 300)|(df['credit_score'] > 850), 'credit_score']= np.nan
df.describe()

,age,income,credit_score,loan_amount
count,180.000000,188.000000,180.000000,200.000000
mean,42.327778,52789.361702,660.288889,151629.500000
std,12.861458,21143.955119,94.675836,86473.742375
min,21.000000,12000.000000,425.000000,5000.000000
25%,31.000000,37925.000000,584.000000,88250.000000
50%,41.000000,51150.000000,665.500000,157450.000000
75%,54.000000,68325.000000,731.750000,205700.000000
max,64.000000,104000.000000,850.000000,448800.000000


In [31]:
#filling nan values
df.isnull().sum()

age                  20
income               12
credit_score         20
loan_amount           0
employment_status    90
married              98
gender               73
approved              0
dtype: int64

In [32]:
df.isna().sum()

age                  20
income               12
credit_score         20
loan_amount           0
employment_status    90
married              98
gender               73
approved              0
dtype: int64

In [33]:
df.age.median()

41.0

In [34]:
#for age we are going with median

# df['age']= df['age'].fillna(df.age.median())

In [35]:
df.columns

Index(['age', 'income', 'credit_score', 'loan_amount', 'employment_status',
       'married', 'gender', 'approved'],
      dtype='object')

In [36]:
#basically we more column with NAN values we can store all column in one varaible run for loop on that then replace with median()

num_colum= ['age', 'income', 'credit_score', 'loan_amount']

for c in num_colum:
    df[c]= df[c].fillna(df[c].median())

df.isna().sum()

age                   0
income                0
credit_score          0
loan_amount           0
employment_status    90
married              98
gender               73
approved              0
dtype: int64

In [37]:
df.head(10)

,age,income,credit_score,loan_amount,employment_status,married,gender,approved
0,41.0,66400.0,665.5,89100.0,NaN,NaN,NaN,Rejected
1,25.0,75700.0,645.0,228600.0,NaN,NaN,NaN,Approved
2,24.0,26400.0,580.0,5000.0,Employed,NaN,NaN,Approved
3,28.0,78500.0,665.5,272600.0,NaN,NaN,NaN,Approved
4,35.0,51150.0,573.0,159000.0,Employed,No,NaN,Rejected
5,26.0,12000.0,574.0,78000.0,NaN,Yes,NaN,Rejected
6,37.0,63200.0,692.0,5000.0,Employed,NaN,Male,Approved
7,41.0,104000.0,784.0,170200.0,Employed,No,NaN,Approved
8,56.0,60500.0,780.0,182400.0,Employed,Yes,Male,Approved
9,23.0,76700.0,665.5,13700.0,NaN,NaN,Female,Approved


In [38]:
#now we have remaining some columns employment_status, married, gender, approved

In [39]:
df[df.age==41]

,age,income,credit_score,loan_amount,employment_status,married,gender,approved
0,41.0,66400.0,665.5,89100.0,NaN,NaN,NaN,Rejected
7,41.0,104000.0,784.0,170200.0,Employed,No,NaN,Approved
15,41.0,82600.0,528.0,283700.0,Self_Employed,Yes,NaN,Approved
19,41.0,64800.0,729.0,157100.0,NaN,No,NaN,Rejected
21,41.0,78300.0,537.0,177600.0,Employed,Yes,NaN,Rejected
29,41.0,21500.0,727.0,112400.0,Employed,Yes,Female,Rejected
32,41.0,81500.0,665.5,54900.0,NaN,NaN,Female,Rejected
35,41.0,60400.0,665.5,160500.0,Unemployed,Yes,Male,Rejected
36,41.0,56400.0,684.0,154500.0,NaN,Yes,Female,Rejected
37,41.0,71400.0,725.0,183100.0,Self_Employed,NaN,Male,Approved


In [40]:
#for all text column we are going with mode
df.married.mode()
df.employment_status.mode()
df.gender.mode

<bound method Series.mode of 0       NaN
1       NaN
2       NaN
3       NaN
4       NaN
       ... 
206     NaN
208    Male
209    Male
210     NaN
211    Male
Name: gender, Length: 200, dtype: object>

In [41]:
#using loop to fill all na values at a time

for c in ['gender', 'employment_status', 'married']:
    df[c]= df[c].fillna(df[c].mode()[0])

df.isnull().sum()

age                  0
income               0
credit_score         0
loan_amount          0
employment_status    0
married              0
gender               0
approved             0
dtype: int64

In [42]:
df.approved.unique()

array(['Rejected', 'Approved'], dtype=object)

In [43]:
#Machines don't understand text so we converting them into 0, 1 using map just like how we used while cleaning the data

df['approved']= df['approved'].map({'Approved':1, 'Rejected':0})
df.head()

,age,income,credit_score,loan_amount,employment_status,married,gender,approved
0,41.0,66400.0,665.5,89100.0,Employed,Yes,Male,0
1,25.0,75700.0,645.0,228600.0,Employed,Yes,Male,1
2,24.0,26400.0,580.0,5000.0,Employed,Yes,Male,1
3,28.0,78500.0,665.5,272600.0,Employed,Yes,Male,1
4,35.0,51150.0,573.0,159000.0,Employed,No,Male,0


In [44]:
#we use one-Hot encoding principle for catagorical data what it does it cretes a dummy columns and drop on 

df= pd.get_dummies(df, columns= ['gender', 'employment_status', 'married'], drop_first= True)
df.head(10)

,age,income,credit_score,loan_amount,approved,gender_Male,employment_status_Self_Employed,employment_status_Unemployed,married_Yes
0,41.0,66400.0,665.5,89100.0,0,True,False,False,True
1,25.0,75700.0,645.0,228600.0,1,True,False,False,True
2,24.0,26400.0,580.0,5000.0,1,True,False,False,True
3,28.0,78500.0,665.5,272600.0,1,True,False,False,True
4,35.0,51150.0,573.0,159000.0,0,True,False,False,False
5,26.0,12000.0,574.0,78000.0,0,True,False,False,True
6,37.0,63200.0,692.0,5000.0,1,True,False,False,True
7,41.0,104000.0,784.0,170200.0,1,True,False,False,False
8,56.0,60500.0,780.0,182400.0,1,True,False,False,True
9,23.0,76700.0,665.5,13700.0,1,False,False,False,True


#### Feature scaling (standardization)
Transform numerical features to comparable scales, often using StandardScaler.
Purpose: help algorithms that are sensitive to feature magnitudes.
 Each value x is converted using the formula z = x - mean\std, where μ is the mean and σ is the standard deviation.

In [45]:
num_column= ['age', 'income', 'credit_score', 'loan_amount']
df[num_column]

,age,income,credit_score,loan_amount
0,41.0,66400.0,665.5,89100.0
1,25.0,75700.0,645.0,228600.0
2,24.0,26400.0,580.0,5000.0
3,28.0,78500.0,665.5,272600.0
4,35.0,51150.0,573.0,159000.0
...,...,...,...,...
206,34.0,28800.0,665.5,162900.0
208,54.0,103000.0,818.0,89900.0
209,55.0,64600.0,688.0,47600.0
210,30.0,40500.0,669.0,290300.0


In [46]:
#we are converting those columns into 0, 1

from sklearn.preprocessing import StandardScaler
scaler= StandardScaler()

#which columns we wanted to convert or standardizing from 0 to  1

df[num_column]= scaler.fit_transform(df[num_column])
df[num_column].head()

,age,income,credit_score,loan_amount
0,-0.098160,0.670402,0.052355,-0.724918
1,-1.412433,1.125193,-0.176488,0.892336
2,-1.494575,-1.285690,-0.902087,-1.699909
3,-1.166007,1.262119,0.052355,1.402438
4,-0.591012,-0.075358,-0.980228,0.085448


In [47]:
df.head()

,age,income,credit_score,loan_amount,approved,gender_Male,employment_status_Self_Employed,employment_status_Unemployed,married_Yes
0,-0.098160,0.670402,0.052355,-0.724918,0,True,False,False,True
1,-1.412433,1.125193,-0.176488,0.892336,1,True,False,False,True
2,-1.494575,-1.285690,-0.902087,-1.699909,1,True,False,False,True
3,-1.166007,1.262119,0.052355,1.402438,1,True,False,False,True
4,-0.591012,-0.075358,-0.980228,0.085448,0,True,False,False,False


In [48]:
#X, y

X= df.drop(columns=['approved'])
y= df['approved']

In [49]:
#finally training model

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

model= LogisticRegression(max_iter=1000)

#train_test_split method
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


#model fitting
model.fit(X_train, y_train)

print(f"Test Accuracy: {model.score(X_test, y_test): .2f}")

Test Accuracy:  0.62


In [52]:
#classification report

from sklearn.metrics import classification_report


y_pred= model.predict(X_test)

report= classification_report(y_test, y_pred)
print(report)

              precision    recall  f1-score   support

           0       0.65      0.74      0.69        23
           1       0.57      0.47      0.52        17

    accuracy                           0.62        40
   macro avg       0.61      0.60      0.61        40
weighted avg       0.62      0.62      0.62        40

